In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import shutil
import zipfile
import hashlib
import json
import time
import platform
import base64

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")

RAW_SOURCE = Path(r"/kaggle/input/datasets/sharianhasan/def-files/power_transformers_fdd_and_rul(1)")
FINAL_SOURCE = Path(r"/kaggle/input/datasets/sharianhasan/def-files/final_5fold(1)")

ROBUSTNESS_REPEATS = 20
BOOTSTRAP = 5000

WORK_DIR = WORK_ROOT / "advanced_run_sklearn190_converged"
CANONICAL_DIR = WORK_ROOT / "final_5fold_canonical"
CODE_ROOT = WORK_ROOT / "dga_advanced_upgrade_converged"


PKG_ROOT = WORK_ROOT / "pydeps_sklearn190"

print("Kernel Python:", sys.version)
print("Platform:", platform.platform())
print("Raw source:", RAW_SOURCE)
print("Canonical source:", FINAL_SOURCE)
print("Raw source exists:", RAW_SOURCE.exists())
print("Canonical source exists:", FINAL_SOURCE.exists())
print("Work directory:", WORK_DIR)
print("Isolated package directory:", PKG_ROOT)


In [ ]:
def show_tree(root, max_items=60):
    root = Path(root)
    if not root.exists():
        print("MISSING:", root)
        return
    print("\nContents of", root)
    count = 0
    for p in sorted(root.rglob("*")):
        try:
            rel = p.relative_to(root)
        except Exception:
            rel = p
        print(" ", str(rel) + ("/" if p.is_dir() else ""))
        count += 1
        if count >= max_items:
            print("  ...")
            break

assert RAW_SOURCE.exists(), f"Raw path does not exist: {RAW_SOURCE}"
assert FINAL_SOURCE.exists(), f"Canonical path does not exist: {FINAL_SOURCE}"

show_tree(RAW_SOURCE if RAW_SOURCE.is_dir() else RAW_SOURCE.parent, 50)
show_tree(FINAL_SOURCE if FINAL_SOURCE.is_dir() else FINAL_SOURCE.parent, 50)

def resolve_raw_archive(source: Path) -> Path:
    source = Path(source)


    if source.is_file() and source.suffix.lower() == ".zip":
        return source


    if source.is_dir():
        exact = list(source.rglob("power_transformers_fdd_and_rul.zip"))
        if exact:
            print("Found expected raw ZIP:", exact[0])
            return exact[0]

        zips = list(source.rglob("*.zip"))
        if len(zips) == 1:
            print("Using the only ZIP inside raw source:", zips[0])
            return zips[0]


        files = [p for p in source.rglob("*") if p.is_file()]
        if not files:
            raise FileNotFoundError(f"No files found under raw source {source}")

        repacked = WORK_ROOT / "power_transformers_fdd_and_rul_repacked.zip"
        if repacked.exists():
            repacked.unlink()

        print("Raw input appears already extracted. Repacking to:", repacked)
        with zipfile.ZipFile(repacked, "w", compression=zipfile.ZIP_DEFLATED) as zf:
            for p in files:
                zf.write(p, arcname=str(p.relative_to(source)))
        return repacked

    raise FileNotFoundError(f"Could not resolve raw dataset from {source}")

RAW_ARCHIVE = resolve_raw_archive(RAW_SOURCE)
print("\nResolved raw archive:", RAW_ARCHIVE)
print("Raw archive size (MB):", round(RAW_ARCHIVE.stat().st_size / 1024**2, 2))


In [ ]:
import base64
import hashlib
import zipfile
import shutil
from pathlib import Path

EMBEDDED_SHA256 = 'c6bab509fe0e50bf1eb0094464316d78680d60606a09760a75a7d5f5908dea25'
_payload = '...'

bundle_zip = WORK_ROOT / "dga_advanced_upgrade_converged.zip"
bundle_zip.write_bytes(base64.b64decode(_payload))

actual_sha = hashlib.sha256(bundle_zip.read_bytes()).hexdigest()
assert actual_sha == EMBEDDED_SHA256, (actual_sha, EMBEDDED_SHA256)

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)

with zipfile.ZipFile(bundle_zip) as zf:
    zf.extractall(WORK_ROOT)


if not (CODE_ROOT / "scripts").exists():
    candidates = [
        p.parent for p in WORK_ROOT.rglob("advanced_common.py")
        if p.name == "advanced_common.py" and p.parent.name == "scripts"
    ]
    if not candidates:
        raise FileNotFoundError("Could not locate scripts/advanced_common.py after extracting embedded bundle")
    detected_root = candidates[0].parent
    print("Detected code root:", detected_root)
    if CODE_ROOT.exists():
        shutil.rmtree(CODE_ROOT)
    shutil.copytree(detected_root, CODE_ROOT)

assert (CODE_ROOT / "scripts" / "advanced_common.py").exists()

print("Advanced code restored:", CODE_ROOT)
print("Bundle SHA256:", actual_sha)


In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path

PKG_ROOT.mkdir(parents=True, exist_ok=True)

def subprocess_pkg_info(import_name, extra_path=None):
    env_check = os.environ.copy()
    if extra_path is not None:
        old = env_check.get("PYTHONPATH", "")
        env_check["PYTHONPATH"] = str(extra_path) + (os.pathsep + old if old else "")
    cmd = [
        sys.executable,
        "-c",
        (
            f"import {import_name}; "
            f"print(getattr({import_name}, '__version__', 'NO_VERSION')); "
            f"print(getattr({import_name}, '__file__', 'NO_FILE'))"
        ),
    ]
    return subprocess.check_output(cmd, text=True, env=env_check).strip()

try:
    existing = subprocess_pkg_info("sklearn", PKG_ROOT)
    existing_lines = existing.splitlines()
    existing_version = existing_lines[0].strip() if existing_lines else None
    existing_file = existing_lines[1].strip() if len(existing_lines) > 1 else ""
except Exception:
    existing_version = None
    existing_file = ""

print("Private sklearn before install:", existing_version)
print("Private sklearn location before install:", existing_file or "<not importable>")


if existing_version != "1.9.0" or str(PKG_ROOT) not in existing_file:
    for p in list(PKG_ROOT.glob("sklearn")) + list(PKG_ROOT.glob("scikit_learn-*.dist-info")):
        if p.is_dir():
            print("Removing stale private directory:", p)
            shutil.rmtree(p)
        elif p.exists():
            print("Removing stale private file:", p)
            p.unlink()

    print("\nInstalling scikit-learn==1.9.0 into:", PKG_ROOT)
    print("This requires Kaggle Internet = ON.")


    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--disable-pip-version-check",
        "--no-deps",
        "--upgrade",
        "--target",
        str(PKG_ROOT),
        "scikit-learn==1.9.0",
    ]
    print("+", " ".join(install_cmd))
    subprocess.run(install_cmd, check=True)

verified = subprocess_pkg_info("sklearn", PKG_ROOT)
print("\nPrivate sklearn verification:")
print(verified)

verified_lines = verified.splitlines()
verified_version = verified_lines[0].strip() if verified_lines else ""
verified_file = verified_lines[1].strip() if len(verified_lines) > 1 else ""

assert verified_version == "1.9.0", (
    "STOP: the experiment runtime does not import scikit-learn 1.9.0. "
    f"Detected {verified_version!r}."
)
assert str(PKG_ROOT) in verified_file, (
    "STOP: sklearn 1.9.0 is not being imported from the private target. "
    f"Imported from: {verified_file}"
)

print("\nSUCCESS: isolated scikit-learn 1.9.0 is ready.")


In [ ]:
import os
import sys
import subprocess

EXPERIMENT_ENV = os.environ.copy()
old_pythonpath = EXPERIMENT_ENV.get("PYTHONPATH", "")
EXPERIMENT_ENV["PYTHONPATH"] = str(PKG_ROOT) + (
    os.pathsep + old_pythonpath if old_pythonpath else ""
)
EXPERIMENT_ENV["MPLBACKEND"] = "Agg"

version_lines = [
    "import sys",
    "import numpy",
    "import pandas",
    "import scipy",
    "import sklearn",
    "import joblib",
    "import matplotlib",
    "print('python=' + sys.version.split()[0])",
    "print('numpy=' + numpy.__version__)",
    "print('pandas=' + pandas.__version__)",
    "print('scipy=' + scipy.__version__)",
    "print('scikit_learn=' + sklearn.__version__)",
    "print('sklearn_file=' + sklearn.__file__)",
    "print('joblib=' + joblib.__version__)",
    "print('matplotlib=' + matplotlib.__version__)",
    "try:",
    "    import shap",
    "    print('shap=' + shap.__version__)",
    "    print('shap_file=' + shap.__file__)",
    "except Exception as e:",
    "    print('SHAP_IMPORT_ERROR=' + repr(e))",
]
version_script = "\n".join(version_lines)

version_output = subprocess.check_output(
    [sys.executable, "-c", version_script],
    text=True,
    env=EXPERIMENT_ENV,
)
print(version_output)

parsed = {}
for line in version_output.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        parsed[key.strip()] = value.strip()

assert parsed.get("scikit_learn") == "1.9.0", (
    "STOP: experiment subprocess is not using scikit-learn 1.9.0. "
    f"Detected: {parsed.get('scikit_learn')!r}"
)

sklearn_file = parsed.get("sklearn_file", "")
assert str(PKG_ROOT) in sklearn_file, (
    "STOP: experiment subprocess imported sklearn from the wrong location: "
    + sklearn_file
)


probe = subprocess.run(
    [sys.executable, "-c", "import shap; print(shap.__version__); print(shap.__file__)"],
    text=True,
    capture_output=True,
    env=EXPERIMENT_ENV,
)

if probe.returncode != 0:
    print("SHAP is not importable in the experiment runtime.")
    print("Installing SHAP privately without replacing scikit-learn dependencies...")
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--disable-pip-version-check",
            "--no-deps",
            "--upgrade",
            "--target",
            str(PKG_ROOT),
            "shap>=0.46",
        ],
        check=True,
    )

    recheck_lines = [
        "import sklearn",
        "import shap",
        "print('sklearn=' + sklearn.__version__)",
        "print('sklearn_file=' + sklearn.__file__)",
        "print('shap=' + shap.__version__)",
        "print('shap_file=' + shap.__file__)",
    ]
    recheck_script = "\n".join(recheck_lines)
    recheck = subprocess.check_output(
        [sys.executable, "-c", recheck_script],
        text=True,
        env=EXPERIMENT_ENV,
    )
    print(recheck)
    assert "sklearn=1.9.0" in recheck
else:
    print("SHAP verification:")
    print(probe.stdout)

print("Dependency verification complete.")


In [ ]:
def sha256(path, block=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

if CANONICAL_DIR.exists():
    shutil.rmtree(CANONICAL_DIR)
CANONICAL_DIR.mkdir(parents=True, exist_ok=True)

def find_canonical_root(root: Path) -> Path:
    root = Path(root)

    if (root / "predictions").is_dir():
        test = root / "predictions" / "pred_Full_temporal_82_h75.csv"
        if test.exists():
            return root

    for p in root.rglob("predictions"):
        if p.is_dir() and (p / "pred_Full_temporal_82_h75.csv").exists():
            return p.parent

    raise FileNotFoundError(f"Could not find canonical predictions under {root}")

FINAL_ZIP = None

if FINAL_SOURCE.is_file() and FINAL_SOURCE.suffix.lower() == ".zip":
    FINAL_ZIP = FINAL_SOURCE

elif FINAL_SOURCE.is_dir():
    exact = list(FINAL_SOURCE.rglob("final_5fold.zip"))
    if exact:
        FINAL_ZIP = exact[0]
    else:
        zips = list(FINAL_SOURCE.rglob("*.zip"))
        if len(zips) == 1:
            FINAL_ZIP = zips[0]

if FINAL_ZIP is not None:
    print("Extracting canonical ZIP:", FINAL_ZIP)
    with zipfile.ZipFile(FINAL_ZIP) as zf:
        zf.extractall(CANONICAL_DIR)

    detected = find_canonical_root(CANONICAL_DIR)
    if detected.resolve() != CANONICAL_DIR.resolve():
        tmp = WORK_ROOT / "_canonical_normalized"
        if tmp.exists():
            shutil.rmtree(tmp)
        shutil.copytree(detected, tmp)
        shutil.rmtree(CANONICAL_DIR)
        shutil.move(str(tmp), str(CANONICAL_DIR))
else:

    detected = find_canonical_root(FINAL_SOURCE)
    print("Canonical input is already extracted:", detected)
    shutil.rmtree(CANONICAL_DIR)
    shutil.copytree(detected, CANONICAL_DIR)

required = [
    CANONICAL_DIR / "predictions" / "pred_Full_temporal_82_h75.csv",
    CANONICAL_DIR / "predictions" / "pred_Full_temporal_82_h100.csv",
    CANONICAL_DIR / "predictions" / "pred_Statistical_28_h75.csv",
    CANONICAL_DIR / "predictions" / "pred_Statistical_28_h100.csv",
]

for p in required:
    assert p.exists(), f"Missing canonical prediction: {p}"

print("\nCanonical root:", CANONICAL_DIR)
print("Canonical prediction files validated.")
print("Raw archive SHA256:", sha256(RAW_ARCHIVE))
if FINAL_ZIP is not None:
    print("Canonical ZIP SHA256:", sha256(FINAL_ZIP))
else:
    print("Canonical input is an extracted directory; source ZIP hash is unavailable.")


In [ ]:
common_text = (CODE_ROOT / "scripts" / "advanced_common.py").read_text()


assert "tol=1e-6" in common_text, "Expected convergence-stabilized tol=1e-6 configuration"
assert "C=30.0" in common_text, "Expected fixed canonical C=30"
assert "class_weight=None" in common_text, "Expected fixed canonical class_weight=None"

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

FEATURES = WORK_DIR / "dense_features"
RESULTS = WORK_DIR / "advanced_results"
FEATURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

env_lines = [
    "import sys",
    "import platform",
    "import numpy",
    "import pandas",
    "import scipy",
    "import sklearn",
    "import joblib",
    "import matplotlib",
    "print('python=' + sys.version.replace(chr(10), ' '))",
    "print('platform=' + platform.platform())",
    "print('numpy=' + numpy.__version__)",
    "print('pandas=' + pandas.__version__)",
    "print('scipy=' + scipy.__version__)",
    "print('sklearn=' + sklearn.__version__)",
    "print('sklearn_file=' + sklearn.__file__)",
    "print('joblib=' + joblib.__version__)",
    "print('matplotlib=' + matplotlib.__version__)",
    "try:",
    "    import shap",
    "    print('shap=' + shap.__version__)",
    "    print('shap_file=' + shap.__file__)",
    "except Exception as e:",
    "    print('shap=IMPORT_ERROR:' + repr(e))",
]
env_script = "\n".join(env_lines)

environment_report = subprocess.check_output(
    [sys.executable, "-c", env_script],
    text=True,
    env=EXPERIMENT_ENV,
)

audit = {
    "raw_source": str(RAW_SOURCE),
    "raw_archive_resolved": str(RAW_ARCHIVE),
    "raw_archive_sha256": sha256(RAW_ARCHIVE),
    "canonical_source": str(FINAL_SOURCE),
    "canonical_root_resolved": str(CANONICAL_DIR),
    "canonical_zip_resolved": str(FINAL_ZIP) if FINAL_ZIP is not None else None,
    "canonical_zip_sha256": sha256(FINAL_ZIP) if FINAL_ZIP is not None else None,
    "advanced_bundle_sha256": EMBEDDED_SHA256,
    "sklearn_install_method": "pip --target, --no-deps",
    "sklearn_target_directory": str(PKG_ROOT),
    "required_sklearn_version": "1.9.0",
    "advanced_lr_C": 30.0,
    "advanced_lr_class_weight": None,
    "advanced_lr_tol": 1e-6,
    "robustness_repeats": ROBUSTNESS_REPEATS,
    "bootstrap": BOOTSTRAP,
}

(WORK_DIR / "KAGGLE_INPUT_AUDIT.json").write_text(json.dumps(audit, indent=2))
(WORK_DIR / "KAGGLE_ENVIRONMENT_LOCK.txt").write_text(environment_report)

print(environment_report)
print(json.dumps(audit, indent=2))

assert "sklearn=1.9.0" in environment_report
assert str(PKG_ROOT) in environment_report


In [ ]:
SCRIPTS = CODE_ROOT / "scripts"

RUN_ENV = EXPERIMENT_ENV.copy()
RUN_ENV["MPLBACKEND"] = "Agg"


RUN_ENV["OMP_NUM_THREADS"] = "2"
RUN_ENV["OPENBLAS_NUM_THREADS"] = "2"
RUN_ENV["MKL_NUM_THREADS"] = "2"
RUN_ENV["NUMEXPR_NUM_THREADS"] = "2"

def run_stage(name, args):
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)
    print("+", " ".join(map(str, args)))

    started = time.time()

    proc = subprocess.Popen(
        list(map(str, args)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=RUN_ENV,
    )

    log_lines = []
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
        log_lines.append(line)

    rc = proc.wait()
    elapsed = time.time() - started

    log_name = name.lower().replace(" ", "_").replace("/", "_")
    (WORK_DIR / f"{log_name}.log").write_text("".join(log_lines))

    if rc != 0:
        raise RuntimeError(f"{name} failed with exit code {rc}")

    print(f"\n{name} completed in {elapsed / 60:.2f} minutes")


In [ ]:
run_stage("01 dense feature generation", [
    sys.executable,
    SCRIPTS / "04_build_dense_features.py",
    "--archive", RAW_ARCHIVE,
    "--output-dir", FEATURES,
])


In [ ]:
run_stage("02 dense representation curve", [
    sys.executable,
    SCRIPTS / "05a_dense_representation_curve.py",
    "--features-dir", FEATURES,
    "--output-dir", RESULTS / "dense_representation",
    "--bootstrap", BOOTSTRAP,
])


In [ ]:
run_stage("03 adaptive stopping", [
    sys.executable,
    SCRIPTS / "05_dense_adaptive_stopping.py",
    "--features-dir", FEATURES,
    "--output-dir", RESULTS / "adaptive",
    "--target-retention", "0.97",
    "--min-class-recall", "0.75",
])


In [ ]:
run_stage("04 formal interaction", [
    sys.executable,
    SCRIPTS / "06_formal_interaction.py",
    "--predictions-dir", CANONICAL_DIR / "predictions",
    "--output-dir", RESULTS / "interaction",
    "--bootstrap", BOOTSTRAP,
])


In [ ]:
run_stage("05 feature family ablation", [
    sys.executable,
    SCRIPTS / "07_feature_family_ablation.py",
    "--features-dir", FEATURES,
    "--output-dir", RESULTS / "ablation",
    "--bootstrap", BOOTSTRAP,
])


In [ ]:
run_stage("06 robustness stress", [
    sys.executable,
    SCRIPTS / "08_robustness_stress.py",
    "--archive", RAW_ARCHIVE,
    "--features-dir", FEATURES,
    "--output-dir", RESULTS / "robustness",
    "--repeats", ROBUSTNESS_REPEATS,
])


In [ ]:
run_stage("07 conformal prediction", [
    sys.executable,
    SCRIPTS / "09_conformal_dense.py",
    "--features-dir", FEATURES,
    "--output-dir", RESULTS / "conformal",
    "--alpha", "0.10",
    "--calibration-fraction", "0.30",
])


In [ ]:
run_stage("08 SHAP horizon stability", [
    sys.executable,
    SCRIPTS / "10_shap_early_stability.py",
    "--features-dir", FEATURES,
    "--output-dir", RESULTS / "shap",
    "--background", "200",
])


In [ ]:
run_stage("09 advanced summary", [
    sys.executable,
    SCRIPTS / "11_summarize_advanced_results.py",
    "--results-dir", RESULTS,
])


In [ ]:
import pandas as pd
from IPython.display import display, Markdown

summary_md = RESULTS / "ADVANCED_RESULTS_SUMMARY.md"
if summary_md.exists():
    display(Markdown(summary_md.read_text()))

candidate_tables = {
    "Dense representation curve":
        RESULTS / "dense_representation" / "dense_representation_curve.csv",

    "Adaptive stopping test result":
        RESULTS / "adaptive" / "adaptive_policy_test_result.csv",

    "Adaptive class-wise result":
        RESULTS / "adaptive" / "adaptive_policy_classwise.csv",

    "Feature-family ablation":
        RESULTS / "ablation" / "feature_family_ablation_75_summary.csv",

    "Robustness summary":
        RESULTS / "robustness" / "robustness_summary.csv",

    "Conformal by horizon":
        RESULTS / "conformal" / "conformal_by_horizon.csv",

    "SHAP stability":
        RESULTS / "shap" / "shap_stability.csv",
}

for title, path in candidate_tables.items():
    print("\n###", title)
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("Not found:", path)

interaction_path = RESULTS / "interaction" / "formal_interaction.json"
print("\n### Formal interaction")
if interaction_path.exists():
    print(json.dumps(json.loads(interaction_path.read_text()), indent=2))
else:
    print("Not found:", interaction_path)


In [ ]:
expected = [
    RESULTS / "dense_representation" / "dense_representation_curve.csv",
    RESULTS / "adaptive" / "adaptive_policy_test_result.csv",
    RESULTS / "interaction" / "formal_interaction.json",
    RESULTS / "ablation" / "feature_family_ablation_75_summary.csv",
    RESULTS / "robustness" / "robustness_summary.csv",
    RESULTS / "conformal" / "conformal_by_horizon.csv",
    RESULTS / "shap" / "shap_stability.csv",
    RESULTS / "ADVANCED_RESULTS_SUMMARY.md",
]

missing = [str(p) for p in expected if not p.exists()]
assert not missing, "Advanced run incomplete. Missing:\n" + "\n".join(missing)


source_copy = WORK_DIR / "analysis_source"
if source_copy.exists():
    shutil.rmtree(source_copy)

shutil.copytree(
    CODE_ROOT,
    source_copy,
    ignore=shutil.ignore_patterns(
        "__pycache__",
        "*.pyc",
        "advanced_run",
        "advanced_outputs",
    ),
)


freeze_text = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True,
)
(WORK_DIR / "KAGGLE_SYSTEM_PIP_FREEZE.txt").write_text(freeze_text)


isolated_report = subprocess.check_output(
    [
        sys.executable,
        "-c",
        (
            "import sklearn, sys; "
            "print('sklearn_version=' + sklearn.__version__); "
            "print('sklearn_file=' + sklearn.__file__); "
            "print('python=' + sys.version.replace(chr(10),' '))"
        ),
    ],
    text=True,
    env=RUN_ENV,
)
(WORK_DIR / "ISOLATED_SKLEARN190_PROOF.txt").write_text(isolated_report)

ZIP_OUT = WORK_ROOT / "advanced_run_sklearn190_converged.zip"
if ZIP_OUT.exists():
    ZIP_OUT.unlink()

shutil.make_archive(
    str(ZIP_OUT.with_suffix("")),
    "zip",
    root_dir=WORK_DIR.parent,
    base_dir=WORK_DIR.name,
)

print("\n" + "=" * 100)
print("SUCCESS")
print("=" * 100)
print("Result folder:")
print(WORK_DIR)
print("\nOUTPUT ZIP:")
print(ZIP_OUT)
print("\nZIP size (MB):", round(ZIP_OUT.stat().st_size / 1024**2, 2))
print("ZIP SHA256:", sha256(ZIP_OUT))
print("\nExperiment scikit-learn proof:")
print(isolated_report)
